In [0]:
dbutils.widgets.removeAll()

In [0]:
from datetime import datetime, timezone
import time
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()

# search all decployed pipelines
all_pipelines = list(w.pipelines.list_pipelines())
pipeline_options = [p.name for p in all_pipelines] if all_pipelines else ["None"]

dbutils.widgets.dropdown("action", "Audit Stale Runs", ["Audit Stale Runs", "Find Jobs by Name", "Trigger Pipeline"], "Select Action")
dbutils.widgets.text("stale_hours_threshold", "2.0", "Stale Threshold (Hours)")
dbutils.widgets.text("name_pattern", "dev", "Name Pattern")
dbutils.widgets.dropdown("pipeline_to_run", pipeline_options[0], pipeline_options, "DLT Pipeline to Run")
dbutils.widgets.dropdown("full_refresh", "False", ["False", "True"], "Full Refresh")

In [0]:
action = dbutils.widgets.get("action")
stale_threshold = float(dbutils.widgets.get("stale_hours_threshold")) # if job work more than X hours, flag it as bug
pattern = dbutils.widgets.get("name_pattern")
selected_pipeline = dbutils.widgets.get("pipeline_to_run")
full_refresh = dbutils.widgets.get("full_refresh") == "True"

print(f"Starting action: {action}")

#======================================================================================
if action == "Audit Stale Runs":
    now_ms = datetime.now(timezone.utc).timestamp() * 1000
    stale_records = []
    
    for run in w.jobs.list_runs(active_only=True):
        duration_hours = (now_ms - run.start_time) / (1000 * 3600)
        if duration_hours > stale_threshold:
            stale_records.append({
                "Run ID": run.run_id,
                "Job ID": run.job_id,
                "Run Name": run.run_name,
                "Duration (Hours)": round(duration_hours, 2),
                "LifeCycle State": run.state.life_cycle_state.value,
                "Run Page URL": run.run_page_url
            })
            
    if stale_records:
        df_stale = pd.DataFrame(stale_records)
        display(df_stale)
    else:
        print(f"OK: No active runs longer than {stale_threshold} hours.")

#======================================================================================
elif action == "Find Jobs by Name":
    matched_jobs = []
    for job in w.jobs.list():
        # take our job in list 
        if pattern.lower() in job.settings.name.lower():
            matched_jobs.append({
                "Job ID": job.job_id,
                "Job Name": job.settings.name,
                "Creator": job.creator_user_name,
                "Created At": datetime.fromtimestamp(job.created_time / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            })
            
    if matched_jobs:
        df_jobs = pd.DataFrame(matched_jobs)
        display(df_jobs)
    else:
        print(f"No jobs matched the pattern: '{pattern}'")

#======================================================================================
elif action == "Trigger Pipeline":

    target_pipeline = None
    for p in all_pipelines:
        if p.name == selected_pipeline:
            target_pipeline = p
            break
    
    if not target_pipeline:
        raise ValueError(f"Pipeline '{selected_pipeline}' not found.")
        
    p_id = target_pipeline.pipeline_id
    print(f"Triggering pipeline '{selected_pipeline}' | (ID: {p_id}) | Is Full Refresh? -> {full_refresh}")
    
    try:
        resp = w.pipelines.start_update(pipeline_id=p_id, full_refresh=full_refresh)
        update_id = resp.update_id
        print(f"Update started successfully. Update ID: {update_id}")
    except Exception as e:
        if "ResourceConflict" in str(e):
            print("Pipeline is already executing an update. Attaching to the latest run...")
            info = w.pipelines.get_pipeline(pipeline_id=p_id)
            update_id = info.latest_updates[0].update_id
        else:
            raise e
    while True:
        status_info = w.pipelines.get_update(pipeline_id=p_id, update_id=update_id)
        current_state = status_info.update.state.value
        print(f"Status: {current_state}")
        
        if current_state == "COMPLETED":
            print("Pipeline execution finished successfully.")
            break
        elif current_state in ["FAILED", "CANCELED"]:
            raise RuntimeError(f"Pipeline terminated with state: {current_state}")
            
        time.sleep(15)